In [0]:
%run "/Workspace/Users/rahulpatel@cyntexa.com/DataEngineering-Project/de_project/src/includes"

In [0]:
# dbutils.widgets.text("catalog","de_dev")

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog}.gold.sales_kpi AS

WITH sales_summary AS (

    SELECT
        YEAR(s.sale_date) AS year,
        MONTH(s.sale_date) AS month,
        s.region,
        p.category,
        c.customer_id,
        c.name AS customer_name,

        SUM(s.sale_amount) AS total_revenue,
        SUM(s.quantity) AS total_quantity,
        COUNT(s.sale_id) AS total_orders

    FROM ${catalog}.silver.sales_scd_1 s

    JOIN ${catalog}.silver.products_scd_2 p
        ON s.product_id = p.product_id

    JOIN ${catalog}.silver.customers_scd_1 c
        ON s.customer_id = c.customer_id

    GROUP BY
        YEAR(s.sale_date),
        MONTH(s.sale_date),
        s.region,
        p.category,
        c.customer_id,
        c.name
)

SELECT

    year,
    month,
    region,
    category,
    customer_id,
    customer_name,

    total_orders,
    total_quantity,
    total_revenue,

    AVG(total_revenue) OVER(PARTITION BY region) AS avg_region_revenue,

    ROW_NUMBER() OVER(
        PARTITION BY year, month
        ORDER BY total_revenue DESC
    ) AS row_num,

    RANK() OVER(
        PARTITION BY year, month
        ORDER BY total_revenue DESC
    ) AS customer_rank,

    DENSE_RANK() OVER(
        PARTITION BY year, month
        ORDER BY total_revenue DESC
    ) AS dense_customer_rank,

    LAG(total_revenue) OVER(
        PARTITION BY region
        ORDER BY year, month
    ) AS previous_period_revenue,

    ROUND(
        (
            (total_revenue -
             LAG(total_revenue) OVER(
                 PARTITION BY region
                 ORDER BY year, month
             ))
            /
            LAG(total_revenue) OVER(
                 PARTITION BY region
                 ORDER BY year, month
             )
        ) * 100,
        2
    ) AS growth_percentage

FROM sales_summary;